# Week 1 · Foundations & Custom Vision with VLMs
### Build Custom AI — SarasAI Live Session 1

**What you will do today**

1. Refresh the mental model: tokens → embeddings → attention → Transformers.
2. Meet the Hugging Face ecosystem: the Hub, `transformers`, `pipeline()`, `datasets`.
3. Load a **Vision-Language Model (VLM)** and make it *see*.
4. Turn the VLM into a **visual defect detector** for product images (MVTec AD).
5. Measure it properly on a **frozen held-out split** — accuracy, precision/recall, confusion matrix.

**Hardware:** everything here runs on a single ~16 GB GPU (Colab T4, RunPod, or a HF Space).

> 📏 **The Evaluation Rule (applies all 4 weeks):** no deliverable is *done* without a measured
> number on a held-out set you never prompt-engineered against. We freeze that set **before** we
> start engineering — you'll see this pattern at the end of the notebook.

---
## 0 · Setup

We install the few libraries we need. `transformers` gives us models, `datasets` gives us data,
`accelerate` handles device placement, and `scikit-learn` will score our detector at the end.

*(On Colab this takes ~1 minute. The `-q` flag keeps the output quiet.)*

In [ ]:
!pip install -q "transformers>=4.50" datasets accelerate pillow scikit-learn matplotlib

Quick sanity check — do we actually have a GPU, and how much memory does it have?
If this prints a T4 with ~15 GB, you are exactly where the course expects you to be.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# T4s (and other pre-Ampere GPUs) have no native bfloat16 — fall back to float16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("Using dtype:", DTYPE)

---
## 1 · Foundations refresher: what a Transformer actually does

Before we touch a VLM, let's rebuild the intuition in three tiny steps, using real tools —
not slides. Every modern LLM/VLM does exactly this:

1. **Tokenize** — split text into sub-word pieces and map each to an integer ID.
2. **Embed** — turn each ID into a vector (a point in a high-dimensional space).
3. **Attend** — let every token *look at* every other token and decide what matters.

Step 1 is fully inspectable. Let's look inside a real tokenizer.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")

text = "The bottle cap is cracked near the seal."
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("IDs:   ", ids)

Notice how words get split into sub-word pieces — the model has never seen "words",
only these pieces. **Everything an LLM knows is expressed as operations on these IDs.**

Now step 2: each ID becomes a learned vector. The *meaning* lives in the geometry —
similar concepts end up close together. Let's verify that claim with a tiny experiment.

In [ ]:
from transformers import AutoModel
import torch.nn.functional as F

# the 135M variant carries the identical lesson at 1/12th the download
# (the SmolLM2 family shares one tokenizer, so the 1.7B tokenizer above pairs fine with the 135M model)
model = AutoModel.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", torch_dtype=torch.float32)
emb = model.get_input_embeddings()          # the embedding table: one vector per token ID

def vec(word):
    ids = tokenizer.encode(word, add_special_tokens=False)
    return emb(torch.tensor(ids)).mean(dim=0)   # average if the word splits into pieces

pairs = [("crack", "fracture"), ("crack", "banana"), ("defect", "flaw")]
for a, b in pairs:
    sim = F.cosine_similarity(vec(a), vec(b), dim=0).item()
    print(f"cos({a!r:12}, {b!r:12}) = {sim:.3f}")

`crack`/`fracture` and `defect`/`flaw` should score clearly higher than `crack`/`banana` —
the embedding space has learned that they mean similar things. *(We'll exploit exactly this
property next week when we build retrieval.)*

Step 3, **attention**, is the Transformer's core trick: each token computes a weighted mix of all
other tokens — "which words should I look at to understand myself?" We won't re-implement it
(the primary source is *Attention Is All You Need*, arXiv:1706.03762), but here is the one-line
summary worth memorizing:

> A Transformer is a stack of layers that repeatedly *mix information across positions*
> (attention) and *transform it per position* (MLP). That's it. LLMs, VLMs, and
> embedding models are all this same machine at different sizes with different inputs.

Free up the memory before we load the real model:

In [ ]:
del model, emb
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("cleaned up")

---
## 2 · The Hugging Face ecosystem in 5 minutes

Three things to internalize about the [Hub](https://huggingface.co):

| Concept | What it is | Why you care |
|---|---|---|
| **Model repo** | Git repo with weights + config + tokenizer | `AutoModel.from_pretrained("org/name")` pulls everything |
| **Model card** | The README of a model | License, intended use, eval numbers — *read it before you ship it* |
| **`pipeline()`** | One-liner inference wrapper | The fastest way to try any of ~40 task types |

The `pipeline` API is the front door. One line, batteries included:

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

reviews = [
    "This kettle broke after two days. The lid hinge snapped clean off.",
    "Absolutely love it — heats fast and feels premium.",
]
for r in classifier(reviews):
    print(r)

That's a full inference stack — tokenizer, model, post-processing — in one call.
When you need control (custom prompts, generation settings, chat templates) you drop one level
down to `AutoModel*` classes. That's what we'll do with the VLM.

---
## 3 · Vision-Language Models: giving the Transformer eyes

A **VLM** bolts a vision encoder onto a language model:

```
image ──► vision encoder ──► visual tokens ─┐
                                            ├──► language model ──► text
text  ──► tokenizer     ──► text tokens  ───┘
```

The image is converted into *tokens in the same space the LLM already understands* — so the LLM
can reason about pixels with the same machinery it uses for words. This is why we teach VLMs
instead of building CNNs from scratch: **one toolchain, one mental model, and you get language
in/out for free** (structured reports, explanations, few-shot instructions).

We'll use **SmolVLM2-2.2B-Instruct** — small enough for a T4, good enough to be useful.

> 💡 *Low VRAM (<6 GB)? Swap the model ID for `HuggingFaceTB/SmolVLM-256M-Instruct` — the code
> is identical.*

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,            # half precision: 2x less memory, fine for inference
    device_map="auto",            # accelerate puts it on the GPU for us
)
print(f"Loaded {MODEL_ID}  ·  {vlm.num_parameters()/1e9:.2f}B parameters")

The **processor** is the multimodal version of a tokenizer: it resizes/normalizes images
*and* tokenizes text, producing one input dict. Conversations use the same chat-message format
you know from LLM APIs — the only new thing is an `image` content block.

Let's write a small helper we'll reuse all session. Note how short it is — the chat template
does the heavy lifting:

In [ ]:
def ask_vlm(image, question, max_new_tokens=200):
    """Ask the VLM one question about one PIL image. Returns the text answer."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm.device, dtype=DTYPE)

    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = processor.decode(out[0], skip_special_tokens=True)
    return answer.split("Assistant:")[-1].strip()

Smoke test with a random image from the web — can it see at all?

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # the classic "two cats" photo
img = Image.open(BytesIO(requests.get(url, timeout=30).content))
display(img.resize((320, 240)))

print(ask_vlm(img, "Describe this image in one sentence."))

---
## 4 · The real task: visual defect detection

**Scenario (this is Capstone Increment 1):** your company manufactures bottles. Quality control
photographs every unit. You must flag defective ones — *and produce a short structured note
per image* that downstream systems (Week 2's RAG, Week 3's Analyst) can consume.

We use **MVTec AD**, the standard industrial anomaly-detection benchmark: real photos of
products with real defects (cracks, contamination, broken parts). We'll work with the
`bottle` category.

*(Dataset mirror on the Hub: `TheoM55/mvtec_anomaly_detection`. Original: mvtec.com — CC BY-NC-SA 4.0,
research/education use.)*

In [ ]:
from huggingface_hub import snapshot_download

# The full mirror is ~5 GB across 15 product categories — we need ONE category.
# snapshot_download with allow_patterns fetches just the bottle folders (~150 MB, ~2 min):
DATA_ROOT = snapshot_download(
    "TheoM55/mvtec_anomaly_detection", repo_type="dataset",
    allow_patterns=["images/train/bottle/*", "images/test/bottle/*"],
)
print("downloaded to:", DATA_ROOT)

> ⚠️ **Know your dataset's conventions before you touch it.** MVTec AD has two quirks that
> would silently break a naive loader:
> 1. The **train split contains only good units** (it was built for anomaly detection, where
>    you train on normal data only). All defective images live in the **test** split.
> 2. The repo also carries **ground-truth segmentation masks** (under `masks/`) — those aren't
>    product photos; our download patterns simply never fetch them.
>
> Note what we did above: we downloaded **files by path pattern**, not "the dataset" — a 5 GB
> corpus became a 150 MB, 2-minute download because we only asked for what we need. The cell
> below is our *data adapter*: a clean `[(PIL image, "good"|"defective"), ...]` interface over
> those files. If you swap mirrors or data sources, only these two cells change — everything
> after them stays the same. That separation between *adapter* and *logic* is a habit worth copying.

In [ ]:
import glob
from collections import Counter
from PIL import Image

def extract_items(root, category="bottle", per_class=30):
    """Adapter: [(image, 'good'|'defective'), ...] — good from train, defects from test.
    Plain sorted-glob over the downloaded files: deterministic, no dataset library needed."""
    good_paths = sorted(glob.glob(f"{root}/images/train/{category}/good/*.png"))[:per_class]
    bad_paths = sorted(p for p in glob.glob(f"{root}/images/test/{category}/*/*.png")
                       if "/good/" not in p)[:per_class]
    return ([(Image.open(p).convert("RGB"), "good") for p in good_paths] +
            [(Image.open(p).convert("RGB"), "defective") for p in bad_paths])

items = extract_items(DATA_ROOT)
counts = Counter(lbl for _, lbl in items)
print("class counts:", dict(counts))          # honest reporting — never assume balance
assert counts["good"] >= 10 and counts["defective"] >= 10, 'adapter found too few images — inspect paths'

Always **look at your data** before modeling. Three good units, three defective:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
shown = {"good": 0, "defective": 0}
for img_, label in items:
    row = 0 if label == "good" else 1
    if shown[label] < 3:
        ax = axes[row][shown[label]]
        ax.imshow(img_); ax.set_title(label); ax.axis("off")
        shown[label] += 1
plt.tight_layout(); plt.show()

---
## 5 · Freeze the held-out set — *before* any prompt engineering

This is the moment that separates a demo from an evaluation. We split the data **now**, put the
test set in a drawer, and do all our prompt iteration on the dev set only. If we peek at test
images while tuning prompts, our final number is fiction.

> 🥶 **Frozen means frozen:** fixed random seed, fixed split, and the test loop runs exactly
> once, at the end.

In [ ]:
import random

random.seed(42)                      # reproducible split — everyone gets the same test set
random.shuffle(items)

split = int(len(items) * 0.5)
dev_set  = items[:split]             # iterate on prompts here
test_set = items[split:]             # touch ONCE, at the very end

print(f"dev: {len(dev_set)}  ·  test: {len(test_set)} (frozen 🔒)")

---
## 6 · Zero-shot defect detection

First attempt: just *ask*. No training, no examples — this is the baseline every custom-AI
project should start with, because it costs nothing and often surprises you.

Two prompt-design decisions worth narrating:
1. **Constrain the output.** We demand a single word, `GOOD` or `DEFECTIVE`, so parsing is trivial.
2. **Give task context.** "Industrial quality inspector" activates the right behavior better
   than a bare question.

In [ ]:
PROMPT_V1 = (
    "You are an industrial quality inspector. Look at this product photo. "
    "Answer with exactly one word: GOOD if the product has no visible defects, "
    "or DEFECTIVE if you can see any damage, contamination, or abnormality."
)

def predict(image, prompt=PROMPT_V1):
    answer = ask_vlm(image, prompt, max_new_tokens=10)
    return "defective" if "DEFECT" in answer.upper() else "good"

# try it on two dev images
for img_, truth in dev_set[:2]:
    print(f"truth={truth:10}  prediction={predict(img_)}")

Now score it on the **dev set** (never the test set — we're still iterating):

In [ ]:
def accuracy(dataset, prompt):
    hits = sum(predict(img_, prompt) == truth for img_, truth in dataset)
    return hits / len(dataset)

acc_v1 = accuracy(dev_set, PROMPT_V1)
print(f"Zero-shot dev accuracy: {acc_v1:.1%}")

---
## 7 · Iterate on the prompt (dev set only)

Typical failure mode: the model calls tiny reflections "defects" (over-sensitive) or misses
subtle contamination (under-sensitive). We iterate by *telling it what defects look like in
this domain* — cheap domain adaptation, no training:

In [ ]:
PROMPT_V2 = (
    "You are an industrial quality inspector for glass bottles. "
    "Defects include: cracks or chips in the glass, contamination inside the bottle, "
    "and a broken or deformed opening. "
    "Normal reflections and lighting variations are NOT defects. "
    "Answer with exactly one word: GOOD or DEFECTIVE."
)

acc_v2 = accuracy(dev_set, PROMPT_V2)
print(f"v1 (generic): {acc_v1:.1%}   →   v2 (domain-specific): {acc_v2:.1%}")

One more rung on the ladder: **few-shot prompting** — instead of *describing* defects, we
*show* the model labeled examples inside the prompt. With a VLM the examples are images:
the message simply interleaves several image blocks with their labels, then the query image.
No training — the "learning" happens in context, and it costs image-tokens on every call:

In [ ]:
def predict_fewshot(image, examples):
    """examples: [(PIL image, 'good'|'defective'), ...] — shown to the model before the query."""
    content = [{"type": "text", "text":
        "You are an industrial quality inspector for glass bottles. Here are labeled examples:"}]
    for ex_img, ex_label in examples:
        content += [{"type": "image", "image": ex_img},
                    {"type": "text", "text": f"Label: {ex_label.upper()}"}]
    content += [{"type": "text", "text": "Now classify this image. Answer GOOD or DEFECTIVE."},
                {"type": "image", "image": image}]

    inputs = processor.apply_chat_template([{"role": "user", "content": content}],
        add_generation_prompt=True, tokenize=True, return_dict=True,
        return_tensors="pt").to(vlm.device, dtype=DTYPE)
    out = vlm.generate(**inputs, max_new_tokens=10, do_sample=False)
    answer = processor.decode(out[0], skip_special_tokens=True).split("Assistant:")[-1]
    return "defective" if "DEFECT" in answer.upper() else "good"

# 1 good + 1 defective exemplar from the DEV set (never from test!)
shots = [next(x for x in dev_set if x[1] == "good"), next(x for x in dev_set if x[1] == "defective")]

fs_eval = [x for x in dev_set if x not in shots]   # exclude the exemplars the model just saw
acc_fs = sum(predict_fewshot(img_, shots) == truth for img_, truth in fs_eval) / len(fs_eval)
print(f"zero-shot v2: {acc_v2:.1%}   →   few-shot (2 exemplars): {acc_fs:.1%}")

Few-shot may or may not beat the domain-specific zero-shot prompt here — small models have
limited in-context capacity for images, and each exemplar costs ~1,500 tokens per call.
**That trade-off (accuracy vs. per-call cost) is exactly the kind of decision your dev set
exists to settle.** For the final test run we commit to the better **zero-shot** prompt — few-shot stays a dev-set experiment here, because its ~3x per-call token cost isn't worth it for this task (and now you have the dev numbers to defend that decision).

Whichever prompt wins on dev is the one we commit to. **Decide now — then stop iterating.**

For the capstone we also need the *structured note* per image. Same technique, JSON output —
this note is exactly what Week 3's Analyst model will consume:

In [ ]:
import json, re

REPORT_PROMPT = (
    "You are an industrial quality inspector for glass bottles. Inspect this photo and "
    "respond ONLY with JSON in this exact format: "
    '{"verdict": "GOOD" or "DEFECTIVE", "defect_type": "<short label or null>", '
    '"description": "<one sentence>"}'
)

def inspect(image):
    raw = ask_vlm(image, REPORT_PROMPT, max_new_tokens=120)
    match = re.search(r"\{.*\}", raw, re.DOTALL)           # tolerate chatter around the JSON
    try:
        return json.loads(match.group()) if match else {"verdict": "PARSE_ERROR", "raw": raw}
    except json.JSONDecodeError:
        return {"verdict": "PARSE_ERROR", "raw": raw}

print(json.dumps(inspect(dev_set[0][0]), indent=2))

---
## 8 · The moment of truth: score the frozen test set

One pass. No more prompt edits after this cell runs. We report the three numbers the
curriculum requires — **accuracy**, and **precision/recall on the defect class** — plus the
confusion matrix, because a single accuracy number hides *which way* the model fails:

- **Precision** (of the units we flagged, how many were truly defective) — low precision =
  we throw away good product.
- **Recall** (of the truly defective units, how many we caught) — low recall = defective
  product ships to customers. *In QC, recall is usually the metric that matters most.*

In [ ]:
import time

FINAL_PROMPT = PROMPT_V2 if acc_v2 >= acc_v1 else PROMPT_V1

t0 = time.perf_counter()
y_true = [truth for _, truth in test_set]
y_pred = [predict(img_, FINAL_PROMPT) for img_, _ in test_set]
elapsed = time.perf_counter() - t0

GPU_PRICE_PER_H = 0.40    # plug in your provider's rate
sec_per_image = elapsed / len(test_set)
print(f"{sec_per_image:.2f} s/image  →  ${GPU_PRICE_PER_H / 3600 * sec_per_image * 1000:.2f} per 1k images")

We timed the loop while it ran — that one extra line gives us the curriculum's secondary
signal (**cost per 1k images**) for free. Now the metrics:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(y_true, y_pred, target_names=None))

cm = confusion_matrix(y_true, y_pred, labels=["good", "defective"])
ConfusionMatrixDisplay(cm, display_labels=["good", "defective"]).plot(cmap="Blues")
plt.title("Held-out test set — frozen split, seed 42")
plt.show()

**How to read your result like a professional:**

- Report *test* numbers, never dev numbers. Include the split seed and set sizes.
- If test ≪ dev, you overfit your prompts to the dev set — that gap is the whole reason
  we froze the split.
- Note the failure direction: false negatives (missed defects) are usually costlier than
  false positives in QC.

This table — metric, set size, seed — is exactly what you submit with **Capstone Increment 1.**

---
## 9 · Production considerations: what changes when this leaves the notebook

| Notebook | Production |
|---|---|
| One image at a time | **Batched** inference; requests queued |
| Any resolution | **Resolution budget** — image tokens dominate cost; downscale/tile deliberately |
| Full VLM for every image | VLM only for *hard* cases; **distill** easy cases into a tiny cheap classifier (→ Week 4) |
| "Looks right" | Accuracy/recall **monitored continuously**; alert on drift |

Back-of-envelope: at ~1,500 image-tokens per photo, a 2.2 B VLM on a T4 handles a few images/sec.
If the line produces 50 units/sec, you either scale horizontally, shrink the model, or triage —
that arithmetic, not accuracy, is usually what decides real architectures.

---
## 10 · Your assignments

**Ungraded warm-up:** pick a *different* MVTec category (e.g. `carpet` or `wood`), run the full loop:
adapter → freeze split → dev iteration → single test pass.

**Graded — Capstone Increment 1:** build the VLM defect detector for the capstone product-image
set. Submit: (1) this notebook adapted to your data, (2) the frozen-split definition (seed + sizes),
(3) test-set accuracy + defect-class precision/recall + confusion matrix, (4) five example
structured JSON notes — they become Week 2's input.

**Read before next week** *(≈45 min)*:
- RAG primary source — Lewis et al. 2020, [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
- [HF multimodal RAG cookbook](https://huggingface.co/learn/cookbook/en/multimodal_rag_using_document_retrieval_and_vlms) (skim — we build a text version live)

*Next week: the model meets your company's knowledge — RAG, KAG & GraphRAG.*